<h1><center>Laboratorio 5: La desperación de Mr. Lepin 🐼</center></h1>

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos</strong></center>

---

### Cuerpo Docente

- Profesores: Pablo Badilla y Diego Cortez
- Auxiliares: Valentina Rojas y Melanie Peña
- Ayudantes: Javiera Arévalo, Tamara Carrasco y Ignacio Reyes


### Equipo: SUPER IMPORTANTE - notebooks sin nombre no serán revisados

- Nombre de alumno 1: Agustin Eduardo Gonzalez Hidalgo
- Nombre de alumno 2: Vicente Ignacio Thiele Muñoz

---

### Reglas

- **Grupos de 2 personas**
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Prohibido copiar.
- Uso de LLM (Copilot, Claude, Antigravity, Cursor, etc.) restringido a consultas, documentación y corrección de errores. 
- **Importante**: **¡Recuerden fijar semillas!** Así podemos reproducir sus resultados.

## Descripción del laboratorio.

### Importamos librerias utiles 😸

In [1]:
!uv add numpy pandas scikit-learn umap-learn plotly

Resolved 126 packages in 1ms
Audited 122 packages in 3ms


In [2]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.base import BaseEstimator, TransformerMixin


def plot_dim_reductions(
    pca_proj: np.ndarray,
    tsne_proj: np.ndarray,
    umap_proj: np.ndarray,
    name: None | str = None,
    colors: None | np.ndarray = None,
) -> go.Figure:
    fig = make_subplots(rows=1, cols=3, subplot_titles=("PCA", "t-SNE", "UMAP"))

    for i, (proj, title) in enumerate(zip([pca_proj, tsne_proj, umap_proj], ["PCA", "t-SNE", "UMAP"], strict=True)):
        temp_fig = px.scatter(
            x=proj[:, 0],
            y=proj[:, 1],
            color=colors.astype(str) if colors is not None else None,
            title=title,
            # showlegend=(i == 0),
        )

        for trace in temp_fig.data:
            trace.showlegend = i == 0
            fig.add_trace(trace, row=1, col=i + 1)

    fig.update_layout(height=400, width=1200, title_text=name)
    return fig

# Segmentación de Clientes en Tienda de Retail 🛍️

<p align="center">
  <img width=300 src="https://s1.eestatic.com/2018/04/14/social/la_jungla_-_social_299733421_73842361_854x640.jpg">
</p>

## 1.1 Cargar Dataset

Mr. Lepin, en una nueva reunión, le cuenta a ud y su equipo que los resultados derivados del análisis exploratorio de datos presentaron una gran utilidad para la empresa y que tiene un gran entusiasmo por continuar trabajando con ustedes.
Es por esto, que Mr. Lepin les pide que cargue y visualicen algunas de las filas que componen el Dataset.
A continuación un extracto de lo parlamentado en la reunión:

    - Usted: Es un gran logro para nuestro equipo que usted haya encontrado excelente el EDA. ¿Qué tiene en mente ahora?
    - Mr. Lepin: Resulta que hace algún tiempo, mientras tomaba un mojito en una reunión de gerentes en Panamá, oí a un *chato* acerca de **LRMFP**, que es un modelo que permite personificar a los clientes a través de la fabricación de distintos atributos que describen a los clientes. Lo encontré es-tu-pendo ñatito. 
    - Usted: Ehh bueno. Investigaremos acerca de este modelo y veremos lo que podemos hacer.

Por ende, su siguiente tarea es calcular **LRMFP** sobre cada cliente y luego hacer un análisis de las características generadas. Para esto, el área de ventas les entrega un nuevo archivo llamado `retail_dataset.pickle`, quien posee los datos del DataFrame original limpios y listos para obtener las características solicitadas por Mr. Lepin.

In [3]:
df_retail = pd.read_pickle(
    "https://github.com/MDS7202/MDS7202/raw/refs/heads/main/recursos/2026-01/labs/lab6/retail_dataset.pickle"
)
df_retail = df_retail.astype(
    {
        "Invoice": str,
        "StockCode": str,
        "Description": str,
        "Customer ID": str,
        "Country": str,
    }
)
df_retail.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## 1.2 Creación de nuevas Caracteristicas [2 Puntos] 

Como ya se les comentó, Mr. Lepin está interesado en obtener las características **LRMFP**, para esto les señala que estas características se construyen en base a las siguientes definiciones:

- **Length (L)**: Intervalo de tiempo, en días, entre la primera y la última visita del cliente. Mientras más grande sea el valor, más fiel es el cliente.

- **Recency (R)**: Indica hace cuánto tiempo el cliente realizó su última compra. Notar que para este caso, mientras más grande es el valor, menos interés posee el usuario para repetir una compra en uno de los locales.

- **Monetary (M)**: El término “monetario” se refiere a la cantidad media de dinero gastada por cada visita del cliente durante el período de observación y refleja la contribución del cliente a los ingresos de la empresa.

- **Frequency (F)**: Se refiere al número total de visitas del cliente durante el periodo de observación. Cuanto mayor sea la frecuencia, mayor será la fidelidad del cliente. 

- **Periodicity (P)**: Representa si los clientes visitan las tiendas con regularidad.

$$Periodicity(n)=std(IVT_1, ..., IVT_n)$$

&nbsp;&nbsp; &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Donde $IVT$ denota el tiempo entre visitas y n representa el número de valores de tiempo entre visitas de un cliente.
 

$$IVT_i=date\_diff(t_{i+1},t_i)$$

En base a las definiciones señaladas, diseñe una función que permita obtener las características **LRMFP** recibiendo un DataFrame como entrada. Para esto, no estará permitido el uso de iteradores; utilicen todas las herramientas que les ofrece `pandas` para realizar esto.

Una referencia que les puede ser útil es el [documento original](https://www.researchgate.net/publication/315979555_LRFMP_model_for_customer_segmentation_in_the_grocery_retail_industry_a_case_study) en donde se propone este método.

**<u>Formato</u> del Resultado Esperado:**

| Customer ID | Length | Recency | Frequency | Monetary | Periodicity |
|------------:|-------:|--------:|----------:|---------:|------------:|
|   12346.0   |    294 |      67 |        46 |   -64.68 |        37.0 |
|   12347.0   |     37 |       3 |        71 |  1323.32 |         0.0 |
|   12349.0   |    327 |      43 |       107 |  2646.99 |        78.0 |
|   12352.0   |     16 |      11 |        18 |   343.80 |         0.0 |
|   12356.0   |     44 |      16 |        84 |  3562.25 |        12.0 |

**Respuesta:**

In [4]:
def custom_features(dataframe_in: pd.DataFrame) -> pd.DataFrame:
    df = dataframe_in.copy()

    features = df.groupby("Customer ID").agg(
        Length=("InvoiceDate", lambda x: (x.max() - x.min()).days),
        Recency=("InvoiceDate", lambda x: (dataframe_in["InvoiceDate"].max() - x.max()).days),
        Frequency=("Invoice", "nunique"),
    )

    total = df.groupby(["Customer ID", "Invoice"])["Price"].sum()

    features["Monetary"] = total.groupby("Customer ID").mean()

    visits = df[["Customer ID", "Invoice", "InvoiceDate"]].drop_duplicates(["Customer ID", "Invoice"])

    periodicity = (
        (visits.groupby("Customer ID")["InvoiceDate"].apply(lambda x: x.diff().dt.days.std()))
        .fillna(0)
        .rename("Periodicity")
    )

    features = features.join(periodicity)

    features = features.round(
        {
            "Monetary": 2,
            "Periodicity": 1,
        }
    )

    return features

In [5]:
custom_features_df = custom_features(df_retail)
custom_features_df.head()

,Length,Recency,Frequency,Monetary,Periodicity
Customer ID,,,,,
12346.0,196,164,11,18.76,36.7
12347.0,37,2,2,81.47,0.0
12348.0,0,73,1,14.39,0.0
12349.0,181,42,3,291.78,101.8
12351.0,0,10,1,49.46,0.0


## 1.3 Pipelines 👷

Finalmente *Mr. Lepin* le pregunta si sería posible realizar un pipeline para realizar una segmentación de los clientes con los nuevos datos generados, a lo que usted responde que **sí** y propone la utilización de k-means para la segmentación.

A continuación siga los pasos requeridos para obtener la segmentación de clientes.

### 1.3.1 Estandarizar Caracteristicas [0.5 puntos]

Construya una clase llamada ``MinMax()`` utilizando ``BaseEstimator`` y ``TransformerMixin`` para realizar una transformación de cada una de las columnas de un DataFrame utilizando ``ColumnTransformer()`` más tarde (tome como referencia el siguiente [enlace](https://sklearn-template.readthedocs.io/en/latest/user_guide.html#transformer)).


 Para esto considere que Min-Max escaler queda dada por la ecuación:

$$MinMax = \dfrac{x-min(x)}{max(x) - min(x)}$$

Con esto buscamos que los valores que componen a las columnas se muevan en el rango de valores $[0, 1]$.

**Respuesta:**

In [6]:
class MinMax(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.columns_ = X.columns
        self.min_ = X.min()
        self.max_ = X.max()
        return self

    def transform(self, X):
        X = pd.DataFrame(X, columns=self.columns_).copy()
        denom = self.max_ - self.min_
        denom[denom == 0] = 1
        return (X - self.min_) / denom

### 1.3.2 Pipelines de Proyecciones [0.5 puntos]

Para comparar técnicas de reducción de dimensionalidad, realice **tres pipelines** distintos sobre los datos **LRMFP** usando los siguientes métodos:
- **PCA**
- **t-SNE**
- **UMAP**

Para cada pipeline, siga estos pasos:
1. Obtenga las características **LRMFP** desde el DataFrame `retail_dataset.pickle` utilizando la función ``custom_features`` creada anteriormente, junto a ``FunctionTransformer()``. Considere esto como el primer paso de su pipeline.
2. En segundo lugar, usando ``ColumnTransformer()``, aplique el MinMax scaler creado por usted sobre todas las columnas generadas en el paso anterior.
3. Finalmente, aplique el método de reducción de dimensionalidad correspondiente (PCA, t-SNE o UMAP) para obtener las 2 componentes más relevantes.

A continuación, grafique las proyecciones obtenidas de las tres técnicas en una sola figura comparativa.

**Respuesta:**

In [7]:
import umap
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

cf = FunctionTransformer(custom_features)
ct = ColumnTransformer(
    transformers=[("minmax", MinMax(), ["Length", "Recency", "Frequency", "Monetary", "Periodicity"])], remainder="drop"
)

pipeline_pca = Pipeline([("function_transformer", cf), ("column_transformer", ct), ("pca", PCA(n_components=2))])
pipeline_tsne = Pipeline(
    [("function_transformer", cf), ("column_transformer", ct), ("tsne", TSNE(n_components=2, random_state=25))]
)
pipeline_umap = Pipeline(
    [("function_transformer", cf), ("column_transformer", ct), ("umap", umap.UMAP(n_components=2, random_state=25))]
)

/home/vixo/universidad/13vosemestre/lab-ciencia-de-datos/Laboratorios-MDS7202/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# Utilice este código para ejecutar las pipelines y graficar.

pca_proj = pipeline_pca.fit_transform(df_retail)
tsne_proj = pipeline_tsne.fit_transform(df_retail)
umap_proj = pipeline_umap.fit_transform(df_retail)

fig = plot_dim_reductions(pca_proj, tsne_proj, umap_proj, name="Reducción de Dimensionalidad", colors=None)
fig.show()

/home/vixo/universidad/13vosemestre/lab-ciencia-de-datos/Laboratorios-MDS7202/.venv/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


### 1.3.3 Análisis de los Loadings de PCA [0.5 puntos]
Antes de continuar con la etapa de clustering, analice los *loadings* (pesos o coeficientes) de las componentes principales obtenidas con PCA. 

Utilice el siguiente tutorial para visualizarlos: https://plotly.com/python/pca-visualization/

- Calcule y reporte los *loadings* de las dos primeras componentes principales.
- Interprete qué características (**LRMFP**) son más relevantes en cada componente.
- Visualice los *loadings* usando un gráfico de barras para cada componente.



In [9]:
# Código para calcular loadings.

feature_names = ["Length", "Recency", "Frequency", "Monetary", "Periodicity"]
pca_loadings = pipeline_pca.named_steps["pca"].components_.T * np.sqrt(
    pipeline_pca.named_steps["pca"].explained_variance_
)
loadings_df = pd.DataFrame(pca_loadings, index=feature_names, columns=["PC1", "PC2"])
print(loadings_df)

                  PC1       PC2
Length       0.343102  0.092210
Recency     -0.184743  0.182791
Frequency    0.017711  0.003550
Monetary    -0.000510  0.000034
Periodicity  0.077855  0.026576


Para la primera componente (PC1) se tiene que Length es la caracteristica más relevante (0.343102), con una dirección positiva, esto nos dice que PC1 captura más la fidelidad de los clientes, mientras que la segunda componente (PC2) Recency es la más relevante con un valor de 0.182791, y también tiene una dirección positiva. Así intrepretamos que esta componente aporta información sobre la actividad reciente de los clientes.

In [10]:
fig = px.scatter(pca_proj, x=0, y=1, title="PCA Loadings", labels={"0": "PC1", "1": "PC2"})

features = loadings_df.index

for i, feature in enumerate(features):
    fig.add_annotation(
        ax=0,
        ay=0,
        axref="x",
        ayref="y",
        x=pca_loadings[i, 0],
        y=pca_loadings[i, 1],
        showarrow=True,
        arrowsize=2,
        arrowhead=2,
        xanchor="right",
        yanchor="top",
    )
    fig.add_annotation(
        x=pca_loadings[i, 0],
        y=pca_loadings[i, 1],
        ax=0,
        ay=0,
        xanchor="center",
        yanchor="bottom",
        text=feature,
        yshift=5,
    )

fig.show()

In [11]:
fig_loadings = go.Figure(
    data=[
        go.Bar(name="PC1", x=feature_names, y=loadings_df["PC1"]),
        go.Bar(name="PC2", x=feature_names, y=loadings_df["PC2"]),
    ]
)
fig_loadings.update_layout(title_text="PCA Loadings Barra", barmode="group")
fig_loadings.show()

### Preguntas sobre loadings:

- ¿Qué son los loadings de PCA?

> Respuesta: Los loadings son valores que nos permiten interpretar que tan relevantes son las variables originales al realizar la reducción de dimensionalidad y el como aportan a estas nuevas componentes.

- ¿Qué información relevante obtiene sobre la estructura de los datos a partir de los *loadings* de PCA?

> Respuesta: Algo que podemos notar es que tanto Frequency como Monetary no aportan mucho a las componentes nuevas, ya que su valores absolutos son muy cercanos a 0. Y lo otro es la gran influencia que tienen Length y Recency en ambas componentes, en donde alcanzan grandes valores en comparación a las demás.

- ¿Existe alguna relación interesante entre las direcciones de las variables?

> Respuesta: En la primera componente se tiene que las dos principales variables que aportan son Length y Recency, pero ambas apuntan hacia direcciones diferentes. Lo que nos dice que al interpretar la primera componente como la fidelidad del cliente esta nos da que existen grandes valores de Length y de Recency que ambos dan información sobre la fidelidad en direcciones contrarias, dando así que un valor alto de PC1 se ve reflejado en un Length alto y un Recency cercano a 0, y en PC2 ambas apuntan hacia la misma dirección, dando a entender esta como la actividad reciente, así podemos decir que un valor alto de PC2 corresponde a un Length y Recency altos, lo que nos dice que clientes antiguos han dejado de comprar recientemente.

## 1.4 Clustering

### 1.4.1 Método del Codo [0.5 puntos]

Utilizando la clase creada para escalamiento, aplique el método del codo para visualizar cuál es el número de clusters que mejor se ajustan a los datos. Realice esto utilizando el algoritmo K-means dentro de un pipeline para un $k \in [1,20]$, donde k representa el número de clusters del k-means. Para la realización de esta sección y la próxima (1.4.2), considere los mismos pasos utilizados para el t-SNE, pero **permutando el algoritmo de reducción de dimensionalidad por k-means.**

**Respuesta:**

In [12]:
from sklearn.cluster import KMeans

inertias = [
    [
        i,
        KMeans(n_clusters=i, random_state=25, n_init="auto").fit(custom_features_df).inertia_,
    ]
    for i in range(1, 20)
]
inertias = pd.DataFrame(inertias, columns=["N° clusters", "Inertia"])
inertias.head()

,N° clusters,Inertia
0,1,3.131333e+08
1,2,2.401046e+08
2,3,8.206422e+07
3,4,5.926700e+07
4,5,5.128567e+07


In [13]:
px.line(
    inertias,
    x="N° clusters",
    y="Inertia",
    title="Método del Codo con K-Means",
    height=600,
)

### Preguntas Método del Codo

- A través del gráfico obtenido, comente y justifique qué valor de k escogería para realizar el k-means.

> Respuesta: Dado el grafico obtenido, podemos aplicar el metodo del codo para obtener el k optimo para k-means, de este modo el valor optimo observado es k = 3, dado que es el punto en donde se marca más el cambio abrupto de la inercia.

- Le fue útil el método del codo para encontrar el número de clusters?

> Respuesta: Sí, porque nos permite visualizar previo al proceso de clustering cual es el numero optimo de clusters, en este caso se ve claramente.

- Si no fue así, ¿qué otros métodos podría haber usado para encontrar un número óptimo de clusters?

> Respuesta: Si no hubiera sido bueno el metodo, habriamos ocupado el coeficiente de Silhouette, debido a que en comparación a otros este mide para cada punto que tan bien esta ubicado dentro de un cluster.

### 1.4.2 Segmentación de Clientes con K-Means 🎁 [1 punto]

Por último, Mr. Lepin, impaciente de no entender lo que usted intenta explicarle, le solicita que por favor muestre algún resultado "visual y entendible" de los grupos encontrados.

En base a la elección de k realizada en la sección anterior, utilice este valor escogido y entrene un modelo de K-means utilizando el mismo pipeline de scikit-learn utilizado anteriormente.

Una vez ajustado los datos, genere una tabla con los promedios (o medianas) para cada uno de los atributos, agrupando estos por el clúster que pertenecen.

Finalmente, construya un heatmap de las características promedio de cada cluster para visualizar y comparar los perfiles de los grupos.

**Estadísticas de Referencia para K=6:**

Ud. debe calcularlas - Varían de ejecución en ejecución.

| Cluster | Length | Recency | Frequency | Monetary | Periodicity | N |
|--------:|-------:|--------:|----------:|---------:|------------:|--:|
| 0 | 258.8 | 45.2 | 76.1 | 1107.7 | 107.6 | 449 |
| 1 | 76.1 | 217.6 | 45.5 | 791.7 | 14.1 | 466 |
| 2 | 368.5 | 4.8 | 2715.0 | 226621.6 | 4.2 | 4 |
| 3 | 85.3 | 45.7 | 65.8 | 1047.0 | 10.5 | 987 |
| 4 | 347.2 | 15.9 | 1658.0 | 35829.3 | 8.0 | 25 |
| 5 | 298.0 | 29.8 | 183.8 | 3639.9 | 32.0 | 1188 |

In [14]:
# Aquí calcule K-Means
pipeline_kmeans = Pipeline(
    [
        ("function_transformer", cf),
        ("column_transformer", ct),
        ("kmeans", KMeans(n_clusters=3, random_state=25, n_init="auto")),
    ]
)

kmeans_labels = pipeline_kmeans.fit_predict(df_retail)

kmeans_df = pd.DataFrame(
    {
        "Length": custom_features_df["Length"],
        "Recency": custom_features_df["Recency"],
        "Frequency": custom_features_df["Frequency"],
        "Monetary": custom_features_df["Monetary"],
        "Periodicity": custom_features_df["Periodicity"],
        "Cluster": kmeans_labels,
    }
)

final_df = (
    kmeans_df.groupby("Cluster")
    .agg(
        Length=("Length", "mean"),
        Recency=("Recency", "mean"),
        Frequency=("Frequency", "mean"),
        Monetary=("Monetary", "mean"),
        Periodicity=("Periodicity", "mean"),
        N=("Cluster", "count"),
    )
    .round(2)
)

final_df.head()

,Length,Recency,Frequency,Monetary,Periodicity,N
Cluster,,,,,,
0,278.68,35.84,8.15,69.16,44.56,1756
1,23.35,250.90,1.60,83.28,3.42,956
2,41.27,54.08,2.11,73.03,5.45,1602


In [15]:
# Utilice la siguiente función para graficar k-means. kmeans_labels = clusters obtenidos por k-means.
plot_dim_reductions(pca_proj, tsne_proj, umap_proj, name="KMeans K=3", colors=kmeans_labels)

In [16]:
# Aquí grafique el Heatmap
px.imshow(
    final_df.drop(columns="N"),
    text_auto=True,
    aspect="auto",
    title="Promedios por Cluster (K-Means K=3)",
    color_continuous_scale="RdBu_r",
)

### Preguntas sobre K-Means: 

- ¿Se separaron bien los distintos clusters en cada visualización? 

> Respuesta: En PCA y t-SNE podemos observar una separación clara de los clusters, mientras que en UMAP la separación es más difusa, los puntos se mezclan y se traslapan entre clusters.

- ¿Es posible observar agrupaciones coherentes?

> Respuesta: Las agrupaciones que si son claras representan grupos coherentes que comparaten caracteristicas en baja dimensionalidad.

- ¿Quedarían mejor más o menos clusters?

> Respuesta: Dadas las visualizaciones de t-SNE y UMAP entendemos que más clusters podrían caracterizar mejor al espacio de baja dimensionalidad.

- ¿K-Means, dada la forma de las proyecciones, será el mejor método para clusterizar este dataset?¿Habrá algún otro mejor?

> Respuesta: Dada las formas de las proyecciones que no son claramente separables entre sí y hay mucho solapamiento, el metodo K-Means puede que no sea el mejor para caracterizar el conjunto en baja dimensionalidad, en este caso nosotros proponemos DBSCAN como un metodo que puede traer mejor resultados para este tipo de comportamiento de los datos.

Y por último:

- Nombre a cada uno de los clusters según el comportamiento de sus miembros (ej. "C1: Compran poco pero con gran valor...") - Si es necesario, ajuste el número de clusters antes de responder.

> Respuestas: C1 = Representa al grupo de compradores recientes que no llevan muchas compras realizadas en la tienda, por lo que tienen poca fidelidad, pero compraron hace poco. El C0 = Representa al grupo de compradores más fieles, pero que no tienen compras tan recientes, aunque estas no llegan a ser tan lejanas. El C2 =  Son los que tienen un punto medio de fidelidad, por lo que se entiende que ya han hecho compras anteriormente, pero no son tan recientes, por lo que no vuelven tanto a la tienda.

Justifique su respuesta y no decepcione a Mr. Lepin.

## 1.5 Detección de Anomalías con DBSCAN [1 punto]
En esta sección, utilizará el algoritmo DBSCAN para identificar posibles anomalías (outliers) en los clientes del retail.

- Puede aplicar DBSCAN sobre las características originales escaladas (**LRMFP**) o sobre alguna de las proyecciones 2D (PCA, t-SNE o UMAP). Justifique su elección en las preguntas al final de la sección.
- Visualice los resultados usando `plot_dim_reductions`, mostrando los clusters y resaltando los outliers (label = -1) en las tres proyecciones (PCA, t-SNE, UMAP).

In [17]:
from sklearn.cluster import DBSCAN

dbscan_labels = DBSCAN(eps=0.85, min_samples=75).fit_predict(umap_proj)

In [18]:
# Utilice este código para graficar. dbscan_labels = clusters/outliers obtenidos por DBSCAN.
fig_dbscan = plot_dim_reductions(
    pca_proj,
    tsne_proj,
    umap_proj,
    name="DBSCAN - Detección de Anomalías",
    colors=dbscan_labels,
)
fig_dbscan.show()

### Preguntas sobre DBSCAN

1. ¿Por qué decidiste usar los datos originales completos o las proyecciones para aplicar DBSCAN? ¿Por qué no usaste la otra opción?

> Respuesta: En este caso decidimos utilizar la proyección de UMAP para aplicar DBSCAN, ya que las separaciones entre puntos nos permite detetar de mejor manera los outliers del conjunto de datos. Y no usamos las otras opciones porque no se pudieron detectar outliers de forma clara.

2. ¿Cómo elegiste los parámetros de DBSCAN (`eps`, `min_samples`)? ¿Probaste diferentes valores? ¿Cómo afectó esto los resultados?

> Respuesta: Fuimos ajustando los valores de eps y min_samples, hasta llegar a eps=0.85, y min_samples = 75, probamos hartos valores, primero variamos eps de 0.3 a 0.9 y min_samples desde 10 hasta 80, pero llegamos a la conclusión de que min_samples tenia que ser un valor alto para generar cluster más densos dentro del eps definido, que fue fijado en 0.85 porque necesitabamos capturar puntos que estuvieran lejanos entre sí, pero no llegar a incluir a todos, de forma de tener outliers que representen una verdadera anomalia.

3. ¿Tienen sentido los outliers encontrados según el contexto del negocio? ¿Qué interpretación le das a estos clientes? Analiza los datos con pandas si es necesario.

> Respuesta: Si, ya que son datos más separados del resto y que no representan una relación aparente entre ellos, por lo que pueden ser omitidos de analisis de caracterización de clientes. Los interpretamos como los clientes dormidos, son los que debemos traer de vuelta, se deben incentivar a que puedan utilizar más la tienda. Pero no tienen un comportamiento muy definido.

# Conclusión
Eso ha sido todo para el lab de hoy, recuerden que el laboratorio tiene un plazo de entrega de una semana. Cualquier duda del laboratorio, no duden en contactarnos por correo, Discord o U-cursos.

![Gracias Totales!](https://i.pinimg.com/originals/65/ae/27/65ae270df87c3c4adcea997e48f60852.gif "bruno")


<br>
<center>
<img src="https://i.kym-cdn.com/photos/images/original/001/194/195/b18.png" width=100 height=50 />
</center>
<br>